In [115]:
# !pip install -U langchain
# !pip install langchain_community
# !pip install -qU pymupdf
# !pip install lark

In [116]:
from dotenv import load_dotenv
load_dotenv()

True

In [128]:
import inspect
import lark
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ContextualCompressionRetriever

FILE_PATH = "./data/virtual_tech_internal_handbook.pdf"

loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
len(docs)

llm = ChatOpenAI(model="gpt-5", temperature=0)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

split_docs = text_splitter.split_documents(docs)


In [123]:
print(docs[0].page_content[:300])

VIRTUALTECH · INTERNAL HANDBOOK
가상Tech
업무 가이드
함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지
구성원이 매일 참고하는 실무 기준
문서 버전
시행일
문서 등급
문의
v1.0
2026-09-01
사내 공개
People Operations
이 문서의 목적
가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.
실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.


In [124]:
def show_metadata(docs):
    if docs:
        print(["[metadata]"])
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

show_metadata(docs)

['[metadata]']
['producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page']

[examples]
producer     : ReportLab PDF Library - (opensource)
creator      : (unspecified)
creationdate : 2026-09-26T11:41:50+09:00
source       : ./data/virtual_tech_internal_handbook.pdf
file_path    : ./data/virtual_tech_internal_handbook.pdf
total_pages  : 9
format       : PDF 1.4
title        : 가상Tech 업무 가이드
author       : 가상Tech People Operations
subject      : RAG 실습용 가상 사내 업무 가이드
keywords     : 
moddate      : 2026-09-26T11:41:50+09:00
trapped      : 
modDate      : D:20260926114150+09'00'
creationDate : D:20260926114150+09'00'
page         : 0


In [125]:
# `pretty_print_docs` 함수는 문서 리스트를 예쁘게 출력하는 헬퍼 함수입니다.
def pretty_print_docs(docs):
    print(
        f"\n{'-'*100}\n".join(
            [f"문서 {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [132]:
embeddings = OpenAIEmbeddings()
faiss_vectorstore = FAISS.from_documents(
    docs,
    embeddings,
)

db = FAISS.from_documents(split_docs, embeddings)

retriever = FAISS.from_documents(docs, OpenAIEmbeddings()).as_retriever()

In [136]:
docs = retriever.invoke("연차를 사용하려면 최소 며칠 전에 신청해야 하나요?")

for doc in docs:
    print(doc.page_content)
    print("=========================================")

가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
08
CHAPTER 07
휴가·비용·복지
예측 가능한 신청 절차로 개인의 휴식과 팀 운영을 함께 지킵니다.
7.1 휴가
구분
신청 시점
승인·증빙
연차
1일 전
팀장 승인, 3일 이상은 2주 전 협의
반차
당일 10:00 전
팀장 승인
병가
가능한 즉시
3일 초과 시 진료 확인서 제출
경조휴가
사유 확인 즉시
People Ops 확인, 사내 규정 일수 적용
7.2 비용 처리
• 법인카드 사용 내역은 영수증과 목적을 지출일로부터 5영업일 안에 ExpenseHub에 등록합니다.
• 건당 10만원 이상 소프트웨어·비품은 구매 전에 팀장 승인을 받습니다. 연간 구독은 IT 자산 검토도 필요합니다.
• 국내 출장 교통·숙박은 출발 5영업일 전 신청합니다. 숙박 한도는 1박 15만원(세금 포함)입니다.
• 택시는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원합니다.
7.3 주요 복지
제도
지원 내용
신청
성장 지원
연 120만원 교육·도서·컨퍼런스
LearnHub에서 사전 승인
건강 지원
연 60만원 운동·상담
월별 영수증 제출
장비 지원
3년 주기 노트북 교체, 모니터 1대
#help-it 티켓
리프레시
근속 3년마다 유급 10영업일
3개월 전 팀과 일정 협의
휴가 중 연락
휴가자는 업무 메시지에 응답할 의무가 없습니다. 인수인계 문서에 대체 담당자,
진행 상태, 긴급 연락 기준을 남기고 Slack 상태와 캘린더를 갱신합니다.
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
09
CHAPTER 08
FAQ와 연락처
헷갈리는 상황은 아래 기준을 우선 적용하고, 필요하면 담당 팀에 문의하세요.
Q. 재택근무일을 다음 주로 이월할 수 있나요?
아니요. 주 2회 기준은 주 단위이며 이월되지 않습니다. 고객 방문이나 건강상 사유의 예외는 팀장과 People Ops가
승인합니다.
Q. 업무용 생성형 AI에 고

In [137]:
prompt = PromptTemplate.from_template(
     """You are an AI language model assistant. 
    Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. 
    By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search. 
    Your response should be a list of values separated by new lines, eg: `foo\nbar\nbaz\n`
    
    #ORIGINAL QUESTION: 
    {question}
    
    #Answer in Korean:
    """
)

llm = ChatOpenAI(model="gpt-5", temperature=0)

chain = (
    {"question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

question = "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?"
response = llm.invoke(question)

chain = prompt | llm




In [138]:
pretty_print_docs(docs)

문서 1:

가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
08
CHAPTER 07
휴가·비용·복지
예측 가능한 신청 절차로 개인의 휴식과 팀 운영을 함께 지킵니다.
7.1 휴가
구분
신청 시점
승인·증빙
연차
1일 전
팀장 승인, 3일 이상은 2주 전 협의
반차
당일 10:00 전
팀장 승인
병가
가능한 즉시
3일 초과 시 진료 확인서 제출
경조휴가
사유 확인 즉시
People Ops 확인, 사내 규정 일수 적용
7.2 비용 처리
• 법인카드 사용 내역은 영수증과 목적을 지출일로부터 5영업일 안에 ExpenseHub에 등록합니다.
• 건당 10만원 이상 소프트웨어·비품은 구매 전에 팀장 승인을 받습니다. 연간 구독은 IT 자산 검토도 필요합니다.
• 국내 출장 교통·숙박은 출발 5영업일 전 신청합니다. 숙박 한도는 1박 15만원(세금 포함)입니다.
• 택시는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원합니다.
7.3 주요 복지
제도
지원 내용
신청
성장 지원
연 120만원 교육·도서·컨퍼런스
LearnHub에서 사전 승인
건강 지원
연 60만원 운동·상담
월별 영수증 제출
장비 지원
3년 주기 노트북 교체, 모니터 1대
#help-it 티켓
리프레시
근속 3년마다 유급 10영업일
3개월 전 팀과 일정 협의
휴가 중 연락
휴가자는 업무 메시지에 응답할 의무가 없습니다. 인수인계 문서에 대체 담당자,
진행 상태, 긴급 연락 기준을 남기고 Slack 상태와 캘린더를 갱신합니다.
----------------------------------------------------------------------------------------------------
문서 2:

가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
09
CHAPTER 08
FAQ와 연락처
헷갈리는 상황은 아래 기준을 우선 적용하고, 필요하면 담당 팀에 문의하세요.
Q. 